# Shape And Cache Test Experiment

Interactive mirror of `tests/test_shapes.py`.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
if repo.name == "nb":
    repo = repo.parent
sys.path.insert(0, str(repo / "src"))
sys.path.insert(0, str(repo))

import torch
from pra_torch.config import PRAConfig
from pra_torch.model import TinyPRALanguageModel
from pra_torch.data import CharTokenizer
from pra_torch.memory import PRASimpleMemoryCache

In [ ]:
tok = CharTokenizer(["hello world"])
cfg = PRAConfig(vocab_size=tok.vocab_size, max_seq_len=16, d_model=32, n_heads=4, n_layers=2)
model = TinyPRALanguageModel(cfg)
x = torch.randint(0, tok.vocab_size, (2, 16))
logits = model(x)

assert logits.shape == (2, 16, tok.vocab_size)
{"input_shape": tuple(x.shape), "logits_shape": tuple(logits.shape), "vocab_size": tok.vocab_size}

In [ ]:
tok = CharTokenizer(["abc", "summary"])
cfg = PRAConfig(vocab_size=tok.vocab_size, max_seq_len=16, d_model=32, n_heads=4, n_layers=2)
model = TinyPRALanguageModel(cfg)
entry = model.encode_reference_to_cache("mem://x", "abc", "summary", tok, "cpu")

assert 0 in entry.layer_kv
assert entry.layer_kv[0].k.ndim == 4
assert entry.layer_kv[0].v.ndim == 4
{"k_shape": tuple(entry.layer_kv[0].k.shape), "v_shape": tuple(entry.layer_kv[0].v.shape)}

In [ ]:
tok = CharTokenizer(["question \u00a4 answer", "secret code red"])
cfg = PRAConfig(
    vocab_size=tok.vocab_size,
    max_seq_len=24,
    d_model=32,
    n_heads=4,
    n_layers=2,
    trigger_threshold=-1.0,
)
model = TinyPRALanguageModel(cfg)
entry = model.encode_reference_to_cache("mem://x", "secret code red", "secret code", tok, "cpu")
cache = PRASimpleMemoryCache()
cache.put(entry)
x = torch.tensor([tok.encode("question \u00a4 answer")[:24]])
model.set_pra_cache(cache)
logits = model(x)

assert logits.shape[0] == 1
{"input_shape": tuple(x.shape), "logits_shape": tuple(logits.shape), "cache_entries": len(cache.entries)}